# Scenario: Legal Research Assistant for a Corporate Compliance Team
 Context
 A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
 legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
 How the RAG Chatbot Fits In
 - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
 - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
 - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
 - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
 - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
 - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
  for non-compliance?”.
 Outcome
 The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives  without needing to manually parse every page.


In [ ]:
# ==========================================================
# LEGAL RESEARCH ASSISTANT (RAG CHATBOT)
# Corporate Compliance Team Use Case
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------

!pip install -q chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
from google.colab import files


# ----------------------------------------------------------
# STEP 2 — Upload and Load Legal PDF Document
# ----------------------------------------------------------

print("Please upload your legal document (example: data_privacy_regulation.pdf)")

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Loading PDF document...")

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted

print("Document Loaded")
print("Total Characters:", len(text))

print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=700, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_collection")
    print("Old collection deleted")
except:
    pass

collection = client.create_collection("legal_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------

print("\nLoading Legal Assistant Model...")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------

def answer_question(query):

    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are a legal research assistant for a corporate compliance team.

Answer the question using ONLY the context below.
Explain the legal text in simple language.

If the answer is not present, say:
"Information not found in the document."

Context:
{context}

Question:
{query}

Answer:
"""

    # Convert text → tokens
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    # Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )

    # Decode tokens → text
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------

print("\n==============================")
print("Legal Research Assistant Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Session ended.")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Please upload your legal document (example: data_privacy_regulation.pdf)


Saving data_privacy_regulation.pdf to data_privacy_regulation (4).pdf
Loading PDF document...
Document Loaded
Total Characters: 2885

Preview:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr

Splitting document into chunks...
Total Chunks Created: 5

Example Chunk:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading Legal Assistant Model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM loaded successfully

Legal Research Assistant Ready
Type 'exit' to stop

Ask a question: tell me about data privacy

Answer:
 Article 3: Rights of Data Subjects  Individuals have the right to access their personal data.

------------------------------------------------------------

